# Talk to Your Database — Colab UI (Qwen3 + QLoRA)

**Runtime:** Colab **Pro** GPU (L4 or roomy T4). Free T4 often OOMs on Qwen3-4B.

Run cells **0 → 5** in order.

## How the adapter is integrated

1. You **upload** the team QLoRA adapter folder (weights + `adapter_config.json`).
2. Cell 3 writes `app/ui_config.json` with `backend = qwen3-4b+adapter` and `adapter_dir` = that folder.
3. When Streamlit starts, `app/backend/models.py`:
   - downloads/loads base **Qwen3-4B-Instruct-2507** from Hugging Face (4-bit),
   - then attaches the adapter with **PEFT** `PeftModel.from_pretrained(base, adapter_dir)`.
4. Sidebar must show **`qwen3-4b+adapter`**. If it shows Qwen2.5, you used the wrong notebook/config.

Open the **Colab proxy URL** from the last cell (not Windows localhost).


## Cell 0 — Clone the GitHub repo

Set `REPO_URL` and `BRANCH`. The UI code (`app/backend/models.py`) must exist on that branch.


In [ ]:
import os, shutil
from pathlib import Path

# === EDIT THESE ===
REPO_URL = "https://github.com/siddhant-192/Group-10-DS-and-AI-Lab-Project.git"
BRANCH = "main"  # must contain full app/backend (UI). Change if UI is only on another branch.

def find_project_root(base=Path("/content")) -> Path:
    hits = sorted(base.glob("**/app/app.py"))
    hits = [h for h in hits if "/.git/" not in str(h).replace("\\", "/")]
    if not hits:
        raise FileNotFoundError("Could not find app/app.py under /content")
    return hits[0].resolve().parents[1]

dest = Path("/content/repo")
if dest.exists():
    shutil.rmtree(dest)
get_ipython().system(f'git clone --depth 1 -b {BRANCH} "{REPO_URL}" /content/repo')

ROOT = find_project_root()
os.environ["PROJECT_ROOT"] = str(ROOT)
print("PROJECT_ROOT =", ROOT)

# Fail early if this branch still has only the stub UI
assert (ROOT / "app" / "backend" / "models.py").is_file(), (
    "app/backend/models.py missing — this branch does not have the Streamlit UI. "
    "Merge the UI PR into main, or set BRANCH to the UI branch (e.g. milestone-6-ui)."
)
text = (ROOT / "app" / "backend" / "models.py").read_text(encoding="utf-8")
assert "PeftModel.from_pretrained" in text, "models.py does not load PEFT adapters"
print("OK: Qwen3+adapter loader present in models.py")


## Cell 1 — Install deps + check GPU


In [ ]:
import os
from pathlib import Path

ROOT = Path(os.environ["PROJECT_ROOT"])
os.chdir(ROOT)
print("cwd", ROOT)

get_ipython().system("pip install -q -r app/scripts/colab-ui-requirements.txt")

import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU (Pro)"
print("CUDA OK:", torch.cuda.get_device_name(0))
print("VRAM (GB, approx):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


## Cell 2 — Upload / locate the QLoRA adapter

**Do this before running the code cell:**

1. In Colab: left sidebar → **Files** → **Upload** (or drag-and-drop).
2. Upload the adapter folder (or a zip of it).
3. If you uploaded a zip, unzip it (example below).
4. Set `ADAPTER_DIR` to the folder that contains `adapter_config.json`.

The base Qwen3 weights are **not** in this folder — only the small QLoRA adapter.


In [ ]:
import os
from pathlib import Path

# === EDIT: path to your uploaded adapter folder ===
ADAPTER_DIR = Path("/content/final_adapter")

# Optional: if you uploaded a zip instead of a folder, uncomment and fix the name:
# get_ipython().system('unzip -q -o /content/final_adapter.zip -d /content/final_adapter')

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f"Adapter folder not found: {ADAPTER_DIR}\n"
        "Upload the QLoRA folder to Colab, then set ADAPTER_DIR to that path."
    )

cfg = ADAPTER_DIR / "adapter_config.json"
weights = list(ADAPTER_DIR.glob("adapter_model*.safetensors")) + list(
    ADAPTER_DIR.glob("adapter_model*.bin")
) + list(ADAPTER_DIR.glob("*.safetensors"))

print("Adapter folder:", ADAPTER_DIR.resolve())
print("Contains adapter_config.json:", cfg.is_file())
print("Weight files found:", [p.name for p in weights[:10]])

if not cfg.is_file():
    raise FileNotFoundError(
        f"Missing adapter_config.json under {ADAPTER_DIR}. "
        "Point ADAPTER_DIR at the folder that holds the PEFT adapter files."
    )

os.environ["ADAPTER_DIR"] = str(ADAPTER_DIR.resolve())
print("ADAPTER_DIR set for later cells.")


## Cell 3 — Write Qwen3 ui_config.json (this is the integration step)

Copies `ui_config.qwen3.example.json` and sets `adapter_dir` to your uploaded folder.

This is what makes the UI load **base Qwen3 + adapter**, not Qwen2.5 and not mock.


In [ ]:
import json
import os
import shutil
from pathlib import Path

ROOT = Path(os.environ["PROJECT_ROOT"])
ADAPTER_DIR = Path(os.environ["ADAPTER_DIR"])
os.chdir(ROOT)

src = ROOT / "app" / "ui_config.qwen3.example.json"
dst = ROOT / "app" / "ui_config.json"
shutil.copy2(src, dst)

cfg = json.loads(dst.read_text(encoding="utf-8"))
cfg["backend"] = "qwen3-4b+adapter"
cfg["model_slug"] = "qwen3-4b-instruct-2507"
cfg["adapter_dir"] = str(ADAPTER_DIR)  # absolute Colab path
cfg["load_4bit"] = True
cfg["max_new_tokens"] = 512
dst.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

print("Wrote", dst)
print(json.dumps(cfg, indent=2))
print()
print("Integration check:")
print("  backend     ->", cfg["backend"])
print("  model_slug  ->", cfg["model_slug"], "(HF base id comes from configs/text2sql_eval_models.json)")
print("  adapter_dir ->", cfg["adapter_dir"], "(PEFT loads this on top of the base)")


## Cell 4 — Demo databases + start Streamlit

Does **not** copy the Qwen2.5 example. Uses the `ui_config.json` from Cell 3.

First model load can take several minutes (download Qwen3 base + attach adapter).


In [ ]:
import json
import os
import socket
import time
from pathlib import Path

ROOT = Path(os.environ["PROJECT_ROOT"])
os.chdir(ROOT)

# Demo DBs (not stored in git)
get_ipython().system("python app/scripts/download_demo_databases.py")

# Confirm config is still Qwen3 (safety)
cfg = json.loads((ROOT / "app" / "ui_config.json").read_text(encoding="utf-8"))
assert cfg.get("backend") == "qwen3-4b+adapter", cfg
assert cfg.get("adapter_dir"), "adapter_dir empty — re-run Cell 2 and Cell 3"
print("Confirmed ui_config backend:", cfg["backend"])
print("Confirmed adapter_dir:", cfg["adapter_dir"])

# Streamlit CORS-safe config for Colab
cfg_dir = ROOT / ".streamlit"
cfg_dir.mkdir(exist_ok=True)
(cfg_dir / "config.toml").write_text(
    "\n".join([
        "[server]",
        "headless = true",
        "enableCORS = false",
        "enableXsrfProtection = false",
        "port = 8501",
        'address = "0.0.0.0"',
        "",
        "[browser]",
        "gatherUsageStats = false",
        "",
    ]),
    encoding="utf-8",
)

os.system("fuser -k 8501/tcp >/dev/null 2>&1")
time.sleep(1)

# Env overrides reinforce Qwen3 + adapter (models.py reads these too)
adapter = cfg["adapter_dir"]
cmd = (
    f"cd {ROOT} && "
    f"MODEL_BACKEND=qwen3-4b+adapter MODEL_SLUG=qwen3-4b-instruct-2507 "
    f"ADAPTER_DIR={adapter} "
    "nohup python -m streamlit run app/app.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--browser.gatherUsageStats false "
    "> /tmp/streamlit_ui.log 2>&1 &"
)
get_ipython().system_raw(cmd)
print("Streamlit launching with Qwen3 + adapter…")

def port_open(port: int = 8501) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False

ok = False
for i in range(90):
    if port_open():
        ok = True
        break
    time.sleep(1)
    if i % 15 == 14:
        print(f"  waiting for port 8501… {i+1}s")

if not ok:
    print("Port 8501 did not open. Log:")
    get_ipython().system("tail -n 80 /tmp/streamlit_ui.log")
else:
    print("Streamlit is ready (background). Next: run Cell 5 for the proxy URL.")
    print("(First question may still take minutes while the model + adapter load.)")


## Cell 5 — Open UI (Colab proxy URL)

Sidebar must show **`qwen3-4b+adapter`**. If not, stop Streamlit and re-check Cells 2–4.


In [ ]:
import socket
import time
from IPython.display import display, HTML
from google.colab.output import eval_js

def port_open(port: int = 8501) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False

if not port_open():
    raise RuntimeError("Port 8501 is closed. Re-run Cell 4 until READY, then this cell.")

time.sleep(1)
url = eval_js("google.colab.kernel.proxyPort(8501)")
print("Expected sidebar backend: qwen3-4b+adapter")
print("Colab proxy URL:")
print(url)
display(HTML(f'<p><a href="{url}" target="_blank" style="font-size:18px">Open Talk to Your Database (Qwen3)</a></p>'))


## Optional — Stop Streamlit


In [ ]:
import os
os.system("fuser -k 8501/tcp >/dev/null 2>&1")
print("Stopped processes on port 8501 (if any).")
